In [ ]:
import os
import csv
import pandas as pd

# Root directory
root_path = r"D:\\mimic_codes\\Report-Generation-and-Disease-Classification\\mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\\mimic-eye"

# Destination CSV file
csv_output_path = r"D:\\mimic_codes\\Report-Generation-and-Disease-Classification\\patients_with_eyegaze_img_bounding_box.csv"

# To store results
rows = []

# Loop over all patient folders
for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)

    if os.path.isdir(patient_path) and patient_folder.startswith("patient_"):
        eyegaze_path = os.path.join(patient_path, "EyeGaze")
        bounding_boxes_csv_path = os.path.join(eyegaze_path, "bounding_boxes.csv")
        cxr_jpg_path = os.path.join(patient_path, "CXR-JPG")
        cxr_meta_path = os.path.join(cxr_jpg_path, "cxr_meta.csv")
        
        # Check if EyeGaze and cxr_meta.csv exist
        if os.path.exists(eyegaze_path) and os.path.exists(cxr_meta_path) and os.path.exists(bounding_boxes_csv_path):
            try:
                df = pd.read_csv(cxr_meta_path)
                if {'study_id', 'dicom_id'}.issubset(df.columns):
                    for _, row in df.iterrows():
                        study_id = row['study_id']
                        dicom_id = row['dicom_id']
                        study_folder_path = os.path.join(cxr_jpg_path, f"s{study_id}")
                        
                        # Check if study folder exists
                        if os.path.exists(study_folder_path) and os.path.isdir(study_folder_path):
                            # Full image path = s{study_id}/{dicom_id}.jpg
                            image_path = os.path.join(study_folder_path, f"{dicom_id}.jpg")
                            
                            if os.path.exists(image_path):
                                rows.append({
                                    "patient_id": patient_folder,
                                    "study_id": study_id,
                                    "dicom_id": dicom_id,
                                    "eyegaze_path": eyegaze_path,
                                    "image_path": image_path,
                                    "bounding_boxes_path": bounding_boxes_csv_path
                                })
                            else:
                                print(f"Image file missing: {image_path}")
                        else:
                            print(f"Study folder missing: {study_folder_path}")
                else:
                    print(f"'study_id' or 'dicom_id' missing in {cxr_meta_path}")
            except Exception as e:
                print(f"Error reading {cxr_meta_path}: {e}")

# Write to CSV
with open(csv_output_path, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=[
        "patient_id", "study_id", "dicom_id", "eyegaze_path", "image_path", "bounding_boxes_path"
    ])
    writer.writeheader()
    writer.writerows(rows)

print(f"Successfully saved {len(rows)} rows into '{csv_output_path}'")


In [15]:
import pandas as pd
import shutil
from pathlib import Path

# 1) Load your master CSV (the one with patient_id, image_path, bounding_boxes_path, etc.)
df = pd.read_csv("patients_with_eyegaze_img_bounding_box.csv")  # <-- replace with your CSV filename

# 2) Create the directory structure
base = Path("dataset")
(images_all := base/"images"/"all").mkdir(parents=True, exist_ok=True)
(labels_all := base/"labels"/"all").mkdir(parents=True, exist_ok=True)

# 3) Copy each image into dataset/images/all/
for img_path in df["image_path"].unique():
    src = Path(img_path)
    dst = images_all / src.name
    shutil.copy(src, dst)

# 4) Merge all per-patient bounding_boxes.csv into one master file
bb_dfs = []
for bb_csv in df["bounding_boxes_path"].unique():
    bb_dfs.append(pd.read_csv(bb_csv))
master_bb = pd.concat(bb_dfs, ignore_index=True)

# 5) Save the combined bounding_boxes.csv into dataset/
master_bb.to_csv(base/"bounding_boxes.csv", index=False)

print("✔️ Created dataset/ with:")
print("   •", images_all, "←", len(list(images_all.glob("*.jpg"))), "images")
print("   •", labels_all, "(empty, for later YOLO .txt files)")
print("   •", base/"bounding_boxes.csv", "←", master_bb.shape[0], "rows")


✔️ Created dataset/ with:
   • dataset\images\all ← 1135 images
   • dataset\labels\all (empty, for later YOLO .txt files)
   • dataset\bounding_boxes.csv ← 18405 rows


In [21]:
import os
import shutil
import pandas as pd
from PIL import Image

# ─── CONFIG ─────────────────────────────────────────────────
master_csv = r"D:\\mimic_codes\\Report-Generation-and-Disease-Classification\\patients_with_eyegaze_img_bounding_box.csv"

DATA_ROOT  = "dataset"
IMG_TRAIN  = os.path.join(DATA_ROOT, "images", "train")
LBL_TRAIN  = os.path.join(DATA_ROOT, "labels", "train")
IMG_VAL    = os.path.join(DATA_ROOT, "images", "val")
LBL_VAL    = os.path.join(DATA_ROOT, "labels", "val")
IMG_TEST   = os.path.join(DATA_ROOT, "images", "test")
LBL_TEST   = os.path.join(DATA_ROOT, "labels", "test")

# split ratios
train_ratio = 0.8   # 80% train
val_ratio   = 0.1   # 10% val
test_ratio  = 0.1   # 10% test

# ─── PREP FOLDERS ────────────────────────────────────────────
for path in (IMG_TRAIN, LBL_TRAIN, IMG_VAL, LBL_VAL, IMG_TEST, LBL_TEST):
    os.makedirs(path, exist_ok=True)

# ─── COORD CONVERSION ────────────────────────────────────────
def xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    xc = (x1 + x2) / 2.0
    yc = (y1 + y2) / 2.0
    w  = x2 - x1
    h  = y2 - y1
    return xc/img_w, yc/img_h, w/img_w, h/img_h

# ─── LOAD & SHUFFLE ──────────────────────────────────────────
df = pd.read_csv(master_csv)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

N = len(df)
i_train = int(N * train_ratio)
i_val   = i_train + int(N * val_ratio)

train_df = df.iloc[:i_train]
val_df   = df.iloc[i_train:i_val]
test_df  = df.iloc[i_val:]

print(f"→ train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

# ─── CACHE ALL BBOX CSVs & BUILD CLASS MAP ────────────────────
bb_cache = {p: pd.read_csv(p) for p in df['bounding_boxes_path'].unique()}
all_names = pd.concat(bb_cache.values())['bbox_name'].unique()
classes = sorted(all_names)
cls2id = {c:i for i,c in enumerate(classes)}

# ─── SPLIT PROCESSING ────────────────────────────────────────
def process_split(split_df, img_dir, lbl_dir):
    for _, row in split_df.iterrows():
        dicom_id = row['dicom_id']
        img_src  = row['image_path']
        bb_path  = row['bounding_boxes_path']

        # copy image
        dst_img = os.path.join(img_dir, f"{dicom_id}.jpg")
        shutil.copyfile(img_src, dst_img)

        # open to get size
        W, H = Image.open(dst_img).size

        # get matching boxes
        boxes = bb_cache[bb_path]
        myb = boxes[boxes['dicom_id'] == dicom_id]
        if myb.empty:
            continue

        # write YOLO .txt
        dst_lbl = os.path.join(lbl_dir, f"{dicom_id}.txt")
        with open(dst_lbl, 'w') as f:
            for _, b in myb.iterrows():
                cid = cls2id[b['bbox_name']]
                x1, x2, y1, y2 = b['x1'], b['x2'], b['y1'], b['y2']
                xc, yc, w, h = xyxy_to_yolo(x1, y1, x2, y2, W, H)
                f.write(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

# run it
process_split(train_df, IMG_TRAIN, LBL_TRAIN)
process_split(val_df,   IMG_VAL,   LBL_VAL)
process_split(test_df,  IMG_TEST,  LBL_TEST)

print("Done!")
print(f"  train imgs: {len(os.listdir(IMG_TRAIN))}, labels: {len(os.listdir(LBL_TRAIN))}")
print(f"     val imgs: {len(os.listdir(IMG_VAL))}, labels: {len(os.listdir(LBL_VAL))}")
print(f"    test imgs: {len(os.listdir(IMG_TEST))}, labels: {len(os.listdir(LBL_TEST))}")


→ train: 908, val: 113, test: 114
Done!
  train imgs: 1021, labels: 974
     val imgs: 113, labels: 109
    test imgs: 114, labels: 109


In [22]:
# PyTorch
import torch

def check_torch_gpu():
    available = torch.cuda.is_available()
    print(f"CUDA available: {available}")
    if available:
        print(f"Device name: {torch.cuda.get_device_name(0)}")
        print(f"Total memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
    else:
        print("Falling back to CPU.")

if __name__ == "__main__":
    check_torch_gpu()


CUDA available: True
Device name: NVIDIA GeForce RTX 3060
Total memory (GB): 12.88


In [3]:
from pathlib import Path
from ultralytics import YOLO

# 1) Train & save best.pt
model = YOLO('yolov8n.pt')
res = model.train(
    data    = 'data.yaml',
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    save    = True,
    project = 'chestxray_runs',
    name    = 'exp1'
)

# 2) Locate best.pt
save_dir     = Path(res.save_dir)               # e.g. "chestxray_runs/exp1"
best_weights = save_dir / 'weights' / 'best.pt'
print("✅ Best weights path:", best_weights)

# 3) Eval each split and print mean metrics
for split in ('train', 'val', 'test'):
    metrics = model.val(
        data  = 'data.yaml',
        model = str(best_weights),
        split = split
    )
    # metrics.box.p, .r, .map50, .f1 are numpy arrays (per-class)
    p_mean   = float(metrics.box.p.mean())     # overall precision
    r_mean   = float(metrics.box.r.mean())     # overall recall
    m50_mean = float(metrics.box.map50.mean()) # overall mAP@0.5
    f1_mean  = float(metrics.box.f1.mean())    # overall F1 score

    print(f"\n— {split.upper():5s} —")
    print(f"Precision: {p_mean:.3f}, Recall: {r_mean:.3f}, mAP@0.5: {m50_mean:.3f}, F1: {f1_mean:.3f}")


New https://pypi.org/project/ultralytics/8.3.121 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.119  Python-3.10.11 torch-2.5.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=chestxray_runs, name=exp1, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt

train: Scanning D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\labels\train.cache... 974 images, 47 backgrounds, 20 corrupt: 100%|██████████| 1021/1021 [00:00<?, ?it/s]

train: D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\images\train\313efdfd-dbe18f7b-a7dad16a-5cbbdeb7-65b4fe7c.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.002555]
train: D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\images\train\3d00eb9f-d57b004b-155401f4-09839a16-33379984.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.002555]
train: D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\images\train\41f4628d-3a6c7bdf-83f3187d-179970df-a99f7a86.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.002555]
train: D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\images\train\4d60e5c5-05733df0-c7d251ea-f46bf386-6b77ff30.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.008058]
train: D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\images\train\584ec4d8-dc0248b5-2e

val: Fast image access  (ping: 0.10.0 ms, read: 21.614.6 MB/s, size: 1787.1 KB)


val: Scanning D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\labels\val.cache... 109 images, 4 backgrounds, 0 corrupt: 100%|██████████| 113/113 [00:00<?, ?it/s]


...

                   all       1001      16214      0.976      0.983      0.985      0.825
    cardiac silhouette        954        954      0.984      0.998      0.992      0.841
         left clavicle        952        952      0.969      0.983      0.989      0.794
left costophrenic angle        954        954      0.931      0.869      0.924      0.576
 left hilar structures        954        954      0.981      0.996      0.992      0.822
  left lower lung zone        954        954       0.98      0.995      0.992      0.862
             left lung        954        954      0.987      0.998      0.993      0.943
    left mid lung zone        954        954      0.985      0.999      0.992      0.864
  left upper lung zone        954        954      0.988      0.997      0.993      0.931
        right clavicle        952        952      0.975      0.985      0.986      0.809
right costophrenic angle        954        954      0.954      0.925      0.953      0.627
right hilar struct

val: Scanning D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\labels\val.cache... 109 images, 4 backgrounds, 0 corrupt: 100%|██████████| 113/113 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:04<00:00,  1.75it/s]


                   all        113       1853      0.984      0.981       0.99      0.836
    cardiac silhouette        109        109      0.987          1      0.995      0.875
         left clavicle        109        109      0.973      0.986      0.993      0.818
left costophrenic angle        109        109       0.97      0.881      0.963      0.605
 left hilar structures        109        109      0.988          1      0.995      0.843
  left lower lung zone        109        109      0.982      0.991      0.992      0.878
             left lung        109        109      0.986      0.991      0.995      0.953
    left mid lung zone        109        109      0.991      0.999      0.995      0.871
  left upper lung zone        109        109      0.986      0.991      0.995      0.932
        right clavicle        109        109      0.991      0.971      0.993      0.821
right costophrenic angle        109        109      0.943      0.927      0.947      0.625
right hilar struct

val: Scanning D:\mimic_codes\Report-Generation-and-Disease-Classification\dataset\labels\test.cache... 109 images, 5 backgrounds, 0 corrupt: 100%|██████████| 114/114 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:05<00:00,  1.45it/s]


                   all        114       1851       0.97      0.965      0.974      0.771
    cardiac silhouette        109        109      0.964      0.972       0.98       0.77
         left clavicle        108        108      0.962       0.93      0.967      0.766
left costophrenic angle        109        109      0.942      0.798      0.866      0.502
 left hilar structures        109        109      0.971      0.982       0.97      0.742
  left lower lung zone        109        109       0.95      0.954      0.979      0.779
             left lung        109        109      0.981          1      0.984      0.884
    left mid lung zone        109        109       0.98      0.991      0.984      0.777
  left upper lung zone        109        109      0.977      0.991      0.986      0.895
        right clavicle        108        108      0.963      0.962      0.969      0.789
right costophrenic angle        109        109      0.965      0.917      0.969      0.584
right hilar struct

In [29]:
import os
import csv
import pandas as pd

# Root directory
root_path = r"D:\\mimic_codes\\Report-Generation-and-Disease-Classification\\mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\\mimic-eye"

# Destination CSV file
csv_output_path = r"D:\\mimic_codes\\Report-Generation-and-Disease-Classification\\patients_with_reflacx_folder.csv"

# To store results
rows = []

# Loop over all patient folders
for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)

    if os.path.isdir(patient_path) and patient_folder.startswith("patient_"):
        reflacx_path = os.path.join(patient_path, "REFLACX", "main_data")
        cxr_jpg_path = os.path.join(patient_path, "CXR-JPG")
        cxr_meta_path = os.path.join(cxr_jpg_path, "cxr_meta.csv")
        
        # Check if REFLACX and cxr_meta.csv exist
        if os.path.exists(reflacx_path) and os.path.exists(cxr_meta_path):
            try:
                df = pd.read_csv(cxr_meta_path)
                if {'study_id', 'dicom_id'}.issubset(df.columns):
                    for _, row in df.iterrows():
                        study_id = row['study_id']
                        dicom_id = row['dicom_id']
                        study_folder_path = os.path.join(cxr_jpg_path, f"s{study_id}")

                        # Image path
                        image_path = os.path.join(study_folder_path, f"{dicom_id}.jpg")
                        
                        # Check if image file exists
                        if os.path.exists(image_path):
                            rows.append({
                                "patient_id": patient_folder,
                                "study_id": study_id,
                                "dicom_id": dicom_id,
                                "image_path": image_path,
                                "reflacx_main_folder_path": reflacx_path
                            })
                        else:
                            print(f"⚠️ Image missing: {image_path}")
                else:
                    print(f"⚠️ 'study_id' or 'dicom_id' missing in {cxr_meta_path}")
            except Exception as e:
                print(f"⚠️ Error reading {cxr_meta_path}: {e}")

# Write to CSV
with open(csv_output_path, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=[
        "patient_id", "study_id", "dicom_id", "image_path", "reflacx_main_folder_path"
    ])
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ Successfully saved {len(rows)} rows into '{csv_output_path}'")


✅ Successfully saved 2652 rows into 'D:\\mimic_codes\\Report-Generation-and-Disease-Classification\\patients_with_reflacx_folder.csv'


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import os

# --- CONFIG ---
reflacx_csv = r"D:\\mimic_codes\\Report-Generation-and-Disease-Classification\\patients_with_reflacx_folder.csv"
weights_path = r"D:\\mimic_codes\\Report-Generation-and-Disease-Classification\\chestxray_runs\\exp1\\weights\\best.pt"

# --- Load Model ---
model = YOLO(weights_path)

# --- Load Patient Image Metadata ---
df = pd.read_csv(reflacx_csv)

# --- Storage ---
patient_bboxes = {}

# --- Predict per Image ---
for idx, row in df.iterrows():
    patient_id = row['patient_id']
    dicom_id = row['dicom_id']
    study_id = row['study_id']
    image_path = row['image_path']
    reflacx_folder = os.path.join(Path(image_path).parents[2], "REFLACX")

    try:
        preds = model.predict(
            source=image_path,
            save=False,
            save_txt=False,
            conf=0.05,
            iou=0.5,
            device=0,
            verbose=False
        )[0]

        # Reset best boxes for this image
        best_per_class = {}

        for box in preds.boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            if cls_id not in best_per_class or best_per_class[cls_id]["confidence"] < conf:
                best_per_class[cls_id] = {
                    "study_id": study_id,
                    "dicom_id": dicom_id,
                    "class_id": cls_id,
                    "class_name": model.names[cls_id],
                    "confidence": conf,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                }

        if patient_id not in patient_bboxes:
            patient_bboxes[patient_id] = {
                "reflacx_folder": reflacx_folder,
                "entries": []
            }

        patient_bboxes[patient_id]["entries"].extend(best_per_class.values())
        print(f"{dicom_id}: {len(best_per_class)} unique class boxes saved")

    except Exception as e:
        print(f"Error on {image_path}: {e}")

# --- Save per-patient CSVs with safe checks ---
# --- Save per-patient CSVs with safe checks ---
for patient_id, data in patient_bboxes.items():
    reflacx_folder = data["reflacx_folder"]
    save_path = os.path.join(reflacx_folder, "reflacx_bounding_boxes.csv")
    os.makedirs(reflacx_folder, exist_ok=True)

    df_all = pd.DataFrame(data["entries"])
    if df_all.empty or 'confidence' not in df_all.columns:
        print(f"Skipped empty or invalid data for patient: {patient_id}")
        continue

    # **Keep only one box per class_id, the one with highest confidence**
        # Group by each image (dicom_id), then for each image
    for dicom_id, df_img in df_all.groupby("dicom_id"):
        # keep only the highest-confidence box per class in that image
        df_best = (
            df_img
            .sort_values("confidence", ascending=False)
            .drop_duplicates(subset=["class_id"], keep="first")
            .sort_values(by=["class_id"])
        )

        # save under REFLACX/<dicom_id>_bboxes.csv
        img_csv = os.path.join(reflacx_folder, f"{dicom_id}_bboxes.csv")
        df_best.to_csv(img_csv, index=False)
        print(f"Saved {len(df_best)} boxes for image {dicom_id} → {img_csv}")



print("\nAll patient CSVs now safely filtered with unique [dicom_id, class_id].")


In [2]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', None)

# -------- Load EyeGaze Data --------
eyegaze_path = r'mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10012498\EyeGaze\fixations.csv'
df_eyegaze = pd.read_csv(eyegaze_path)

# -------- Normalize EyeGaze --------
FRAME_DURATION = 1 / 30.0  # Approximate duration for each frame (assuming 30fps)

df_eyegaze_normalized = pd.DataFrame({
    'start_time': df_eyegaze['Time (in secs)'],
    'end_time': df_eyegaze['Time (in secs)'] + FRAME_DURATION,
    'x': df_eyegaze['X_ORIGINAL'],
    'y': df_eyegaze['Y_ORIGINAL'],
    'pupil_size': df_eyegaze[['LPD', 'RPD']].mean(axis=1, skipna=True)
})

df_eyegaze_normalized.dropna(subset=['x', 'y'], inplace=True)

# -------- Load REFLACX Data --------
reflacx_path = r'mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10020740\REFLACX\main_data\P300R446982\fixations.csv'
df_reflacx = pd.read_csv(reflacx_path)

# -------- Normalize REFLACX --------
df_reflacx_normalized = pd.DataFrame({
    'start_time': df_reflacx['timestamp_start_fixation'],
    'end_time': df_reflacx['timestamp_end_fixation'],
    'x': df_reflacx['x_position'],
    'y': df_reflacx['y_position'],
    'pupil_size': df_reflacx['pupil_area_normalized']
})

# -------- View Result --------
print("✅ Normalized EyeGaze sample:")
print(df_eyegaze_normalized.head())

print("\n✅ Normalized REFLACX sample:")
print(df_reflacx_normalized.head())


✅ Normalized EyeGaze sample:
   start_time  end_time     x     y  pupil_size
0     0.42114  0.454473  1083  1197   19.122015
1     0.77621  0.809543  1089  1577   18.735680
2     1.34250  1.375833  1159   868   17.947215
3     1.55273  1.586063  1777  1671   18.454870
4     1.87653  1.909863  2190  2157   18.399595

✅ Normalized REFLACX sample:
   start_time  end_time     x     y  pupil_size
0       0.092     0.216  1265  1057       1.046
1       0.243     0.772  1464  1088       1.071
2       0.797     0.975  1244   928       0.891
3       1.002     1.124  1315   664       0.670
4       1.153     1.391  1558   628       0.631
